# 🔥 Détection et Segmentation des Incendies par Vision par Ordinateur (CNN / U-Net)
**Responsable :** Salim — *Responsable Vision par Ordinateur (CNN) & Support Data*  
**Architecture choisie :** `FireUNet` (CNN Encodeur-Décodeur Multi-Tâches : Détection Binaire + Segmentation Spatiale de la Zone Touchée)

---
### Récapitulatif des Missions Couvertes :
1. **Tâche 1 (Prétraitement des images satellites)** : Nettoyage des images corrompues, amélioration du contraste atmosphérique (CLAHE), redimensionnement (`128x128`), extraction des masques spectraux et augmentation de données synchronisée (flips, rotations, variations photométriques, bruit gaussien).
2. **Tâche 2 (Entraînement du modèle CNN U-Net)** : Entraînement conjoint avec fonction de perte hybride (*Binary Cross-Entropy + Dice Loss*) pour détecter la présence du feu et segmenter pixel par pixel la surface touchée.
3. **Tâche 3 (Sauvegarde `.pkl` / `.h5` & Fonction `predict(image_path)`)** : Sérialisation du modèle final dans `models/fire_cnn_unet.pkl` et `models/fire_cnn_unet.h5`, et implémentation de `predict(image_path)` retournant **Oui/Non** + **Localisation complète** (secteur cardinal, coordonnées du barycentre, boîte englobante, GPS estimé et % de surface brûlée).
4. **Mission Secondaire (Support Data)** : Génération du dataset global synchronisé (`dataset/global_fire_weather_dataset.csv`) pour Moundir (RNN/LSTM) et l'Arbre de Décision (`main.py` / Streamlit).

In [ ]:
import json
from pathlib import Path
import cv2
import numpy as np
import torch

from preprocessing import (
    clean_dataset_directory,
    preprocess_image,
    extract_fire_segmentation_mask,
    augment_image_and_mask,
)
from model_cnn import FireUNet, train_fire_cnn, load_trained_model
from predict import predict
from data_support import download_and_prepare_kaggle_dataset, build_global_team_dataset

print('PyTorch version:', torch.__version__)

## 1. Tâche 1 : Téléchargement Kaggle, Nettoyage, Redimensionnement et Augmentation

In [ ]:
train_samples, val_samples = download_and_prepare_kaggle_dataset(
    max_kaggle_per_class=180,
    include_satellite_masks=True,
    seed=42,
)
print(f'Échantillons Entraînement : {len(train_samples)} | Validation : {len(val_samples)}')

## 2. Tâche 2 : Entraînement du Modèle CNN Multi-Tâches (`FireUNet`) et Sauvegarde (`.pkl` & `.h5`)

In [ ]:
model, history = train_fire_cnn(
    train_samples=train_samples,
    val_samples=val_samples,
    epochs=10,
    batch_size=16,
    lr=1.2e-3,
    base_filters=16,
    pkl_save_path='models/fire_cnn_unet.pkl',
    h5_save_path='models/fire_cnn_unet.h5',
)

## 3. Tâche 3 : Test de la Fonction `predict(image_path)`

In [ ]:
sample_fire_img = next(p for p, lbl, _ in val_samples if lbl == 1)
result = predict(sample_fire_img, save_visualization='outputs/notebook_fire_pred.png')

print('Est-ce qu\'il y a un feu ? ->', result['fire_detected'])
print('Confiance                  ->', result['confidence'])
print('Localisation               ->', result['location'])
print('Résumé                     ->', result['summary'])

## 4. Mission Secondaire : Préparation du Dataset Global pour Moundir (LSTM) & Système d'Alerte

In [ ]:
csv_path = build_global_team_dataset(
    samples=val_samples + train_samples,
    output_csv='dataset/global_fire_weather_dataset.csv',
    max_rows=200,
)
print('Dataset global sauvegardé dans :', csv_path)